# 实验 05R：逐 cue IPR 匹配的方法预跑

这不是 05A 的修复。05A 的总体主效应因组级 IPR 匹配失败而保持“不可检验”。05R 使用已经看过的旧 seeds `0..7`，只检查一个事后发现的方法问题：

> 在每条 cue 的第一步竞争程度都精确相同时，softmax 与 sparsemax 的秩坍缩差异是否仍随竞争档位发生方向反转？

每条 cue 分别为两种方法求解 $\alpha$，冻结 $\mathrm{IPR}\in\{4,16,64\}$。总预算为 $8\times32\times3\times3=2304$ 个方法对。旧 seeds 只能决定是否值得用全新 seeds `100..107` 开实验 06，不能提供确认性结论。

## a. 输入与版本固定

Notebook 固定到提交 `a3ab916`。它下载实验 05 的公共 Jacobian 核心、05R 实现和冻结 protocol，并逐一核对 SHA-256。任何不一致都会停止。

In [ ]:
import hashlib
import importlib.util
from pathlib import Path
import subprocess
import sys
import urllib.request

required = {
    "torch": "torch",
    "entmax": "entmax==1.3",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

CODE_REV = "bdab7586475c574b049ec8e7595559cde29ec5fd"
BASE = Path("/content/hopfield-dynamic-geometry")
SOURCES = {
    "experiment_05_multimemory_rank_collapse.py": "6e0fb9b3728600dbb1bf2812acde3df8fcc1f15b31cc6055cee2cb39557402ce",
    "experiment_05R_per_cue_ipr_matching.py": "c71cc84ae2bc65370f3afb7b0e61a4db8988fe73ab5a8144cf5813027f8bc1f7",
    "experiment_05R_per_cue_ipr_matching_protocol.md": "aa6563e893889fc251e75cbbcf29c34dc069006296f3a6b9dc932f930f1ed7d8",
}
BASE.mkdir(parents=True, exist_ok=True)
raw_root = f"https://raw.githubusercontent.com/Heptazero/nn-labs/{CODE_REV}/representation-geometry/experiments/hopfield-dynamic-geometry"
for relative, expected in SOURCES.items():
    target = BASE / Path(relative).name
    urllib.request.urlretrieve(f"{raw_root}/{relative}", target)
    actual = hashlib.sha256(target.read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f"SHA-256 mismatch for {relative}: {actual}")
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
print("verified source revision:", CODE_REV)

## b. 变换：每条 cue 独立匹配第一步竞争

先固定 $s=X^\top x_0/N$，再分别求解

$$\mathrm{IPR}\bigl(T(\alpha s)\bigr)=q,\qquad q\in\{4,16,64\}.$$

求解器从 $\alpha=0$ 出发倍增上界，再二分 80 次。两种方法各自相对目标的误差和方法间误差都必须不超过 0.5%。最高 score 并列造成不可达，或者任一方法求解失败时，整对 cue 对称删除。求解不读取终态和 Jacobian。

## c. 输出：同一完整记忆库中的累计 Jacobian

对每个保留方法对计算

$$A_t=DF(x_t),\qquad J_t=A_{t-1}\cdots A_0,\qquad G_t=J_t^\top J_t.$$

方向存活率仍以 $t=1$ 为入口：

$$d_t=\frac{r_{eff}(t)-1}{r_{eff}(1)-1},\qquad
\mathrm{dimension\_auc}=\operatorname{mean}_{t=1}^{12}d_t.$$

每个 seed 和竞争档位先平均全部 32 个目标与三个噪声率，再计算

$$\Delta_q=\mathrm{AUC}_{sparsemax,q}-\mathrm{AUC}_{softmax,q},\qquad
C=\Delta_{middle}-\frac{\Delta_{low}+\Delta_{high}}2.$$


In [ ]:
from experiment_05R_per_cue_ipr_matching import PerCuePilotConfig, run_pilot

config = PerCuePilotConfig()
output_dir = Path("/content/experiment_05R_pilot")
summary = run_pilot(config, output_dir)
summary

## d. 必要性：先审计方法门，再看反转

05R 依次要求：保留方法对至少 95%；最大 IPR 误差不超过 0.5%；七项数值自检对每个 seed 都通过且谱完整；最后才检查 `abs(mean(C)) >= 0.05`。描述性 bootstrap 不能替代这些门，也不能写成确认性证据。

In [ ]:
import json
import pandas as pd
from IPython.display import display

pairs = pd.read_csv(output_dir / "pair_status.csv")
solver = pd.read_csv(output_dir / "solver_results.csv.gz")
checks = pd.read_csv(output_dir / "self_checks.csv")
seed_level = pd.read_csv(output_dir / "seed_level_auc.csv")
interaction = pd.read_csv(output_dir / "seed_interaction.csv")

display(pairs.groupby(["level", "pair_status"]).size().rename("count").reset_index())
display(seed_level)
display(interaction)
print("all numerical checks passed:", bool(checks["passed"].all()))
print(json.dumps(summary, indent=2, ensure_ascii=False))

assert summary["pair_attainability_gate_passed"]
assert summary["ipr_match_gate_passed"]
assert summary["numerical_gate_passed"]

## e. 失败边界与自动判定

如果前三个方法门失败，05R 不能解释交互。如果前三门通过但 `abs(mean(C)) < 0.05`，说明逐 cue 匹配可行，却没有足够信号投入全新 seeds，路线停止。只有全部通过，才允许预注册实验 06。无论结果如何，05B 和二维图册都不在本 notebook 中运行。

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(output_dir / "main_figure.png")))
print((output_dir / "conclusion.md").read_text())

## f. 下载原始产物

压缩包包含逐 cue 求根表、对称删失表、完整 Jacobian 指标与谱、seed 级 $\Delta/C$、描述性 bootstrap、图和自动判定。

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    "/content/experiment_05R_pilot_artifacts", "zip", root_dir=output_dir
)
print("archive:", archive)
files.download(archive)